In [18]:
import osmnx as ox
import networkx as nx
import pickle
from pyrosm import OSM
import pathPlanner

In [ ]:

# Load network + node data
osm = OSM("your_filtered.pbf")
edges, nodes = osm.get_network(network_type="driving", nodes=True)

# Build networkx graph
G = nx.DiGraph()

for _, row in nodes.iterrows():
    G.add_node(row["id"], x=row.geometry.x, y=row.geometry.y)

for _, row in edges.iterrows():
    u, v = row["u"], row["v"]
    oneway = row.get("oneway", False)
    attrs = {
        "length": row.get("length", 1),
        "highway": row.get("highway", "unknown"),
        "geometry": row.get("geometry")
    }
    G.add_edge(u, v, **attrs)
    if not oneway:
        G.add_edge(v, u, **attrs)

In [ ]:

# Assume G is your directed graph (DiGraph)
# Find the strongly connected components (SCCs)
scc = list(nx.strongly_connected_components(G))

# Find the largest strongly connected component (SCC)
largest_scc = max(scc, key=len)

# Remove all nodes not in the largest SCC
nodes_to_remove = set(G.nodes) - largest_scc
G.remove_nodes_from(nodes_to_remove)


In [ ]:
with open("/media/nils/Nils_Data/MIT-Hackathon/road_graph_cleared.pkl", "wb") as f:
    pickle.dump(G, f)

In [14]:
with open("/media/nils/Nils_Data/MIT-Hackathon/road_graph_cleared.pkl", "rb") as f:
    G = pickle.load(f)

In [19]:

# Set the CRS manually (WGS84)
G.graph["crs"] = "EPSG:4326"
pathPlanner = pathPlanner.PathPlanner(G)

orig = ox.nearest_nodes(G, 8.421517, 49.016828)
dest = ox.nearest_nodes(G, 13.400381, 52.518980)

print("solving...")
path = pathPlanner.calculate_path(orig,dest)
print(path)

solving...
[253787472, 253787473, 253787474, 2210372883, 253787475, 253787476, 2207971460, 253787477, 2585498947, 1524672033, 253787479, 2207971463, 2209550145, 332775422, 332775419, 253787480, 51238556, 2207953595, 51238552, 1595257150, 1595257151, 1580179754, 1580179759, 2401872918, 1580179765, 1580179770, 1352825182, 1432007575, 1580179799, 1432007577, 1352825181, 1580179802, 1595257152, 1352825180, 2207953604, 1352825179, 1595257153, 1432007583, 1595257155, 1352825178, 1595257156, 1352825177, 1595257157, 2095290119, 2095290127, 2207953606, 1352825176, 2207953608, 1580179807, 2207953609, 1352825175, 1352825174, 2207953611, 1352825173, 1580179814, 1352825172, 1580179820, 1352825171, 1580179824, 2207953613, 1352825170, 1352825169, 2207953615, 1580179826, 2207953617, 1580179829, 1352825168, 1352825167, 1352825166, 1352825165, 2207953619, 1352825164, 1352825163, 1352825162, 1580179834, 1398426623, 1580179837, 1352825161, 1580179840, 1352825160, 1580179864, 1580179871, 1580179878, 220795